# Titanic — Exploratory Data Analysis

A systematic exploration of the Titanic dataset covering:
1. Load & inspect data
2. Missing values analysis
3. Overall survival rate
4. Survival by Sex, Pclass, Embarked
5. Age distribution by survival
6. Fare distribution
7. Correlation heatmap
8. Family size analysis
9. Title extraction analysis

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Pretty defaults
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 5)

# Resolve data path relative to this notebook
ROOT = pathlib.Path('..').resolve()
TRAIN_PATH = ROOT / 'data' / 'raw' / 'train.csv'
TEST_PATH  = ROOT / 'data' / 'raw' / 'test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print('Train:', train.shape, '  Test:', test.shape)

## 1. Load & Inspect Data

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe().round(2)

## 2. Missing Values Analysis

In [ ]:
missing = train.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(1)
missing_df = pd.DataFrame({'Count': missing, 'Percent (%)': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0]
print(missing_df)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(train.isnull().T, cbar=False, yticklabels=True, cmap='viridis', ax=ax)
ax.set_title('Missing Values Heatmap (yellow = missing)', fontsize=13)
ax.set_xlabel('Passenger index')
plt.tight_layout()
plt.show()

## 3. Overall Survival Rate

In [ ]:
survived_counts = train['Survived'].value_counts()
labels = ['Did Not Survive', 'Survived']
colors = ['#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(survived_counts, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Survival Distribution')

sns.countplot(x='Survived', data=train, palette=colors, ax=axes[1])
axes[1].set_xticklabels(['Did Not Survive', 'Survived'])
axes[1].set_title('Survival Counts')
axes[1].set_xlabel('')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2, p.get_height() + 5),
                     ha='center', fontsize=12)

plt.suptitle(f'Overall Survival Rate: {train["Survived"].mean():.1%}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Survival by Sex, Pclass, and Embarked

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# By Sex
sex_surv = train.groupby('Sex')['Survived'].mean().reset_index()
sns.barplot(x='Sex', y='Survived', data=sex_surv, palette=['#3498db', '#e91e8c'], ax=axes[0])
axes[0].set_title('Survival Rate by Sex')
axes[0].set_ylabel('Survival Rate')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1%}', (p.get_x() + p.get_width()/2, p.get_height()+0.01),
                     ha='center')

# By Pclass
pclass_surv = train.groupby('Pclass')['Survived'].mean().reset_index()
sns.barplot(x='Pclass', y='Survived', data=pclass_surv, palette='Blues_d', ax=axes[1])
axes[1].set_title('Survival Rate by Passenger Class')
axes[1].set_ylabel('Survival Rate')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1%}', (p.get_x() + p.get_width()/2, p.get_height()+0.01),
                     ha='center')

# By Embarked
emb_surv = train.groupby('Embarked')['Survived'].mean().reset_index()
sns.barplot(x='Embarked', y='Survived', data=emb_surv, palette='Set2', ax=axes[2])
axes[2].set_title('Survival Rate by Embarked Port')
axes[2].set_ylabel('Survival Rate')
axes[2].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for p in axes[2].patches:
    axes[2].annotate(f'{p.get_height():.1%}', (p.get_x() + p.get_width()/2, p.get_height()+0.01),
                     ha='center')

plt.tight_layout()
plt.show()

# Survival rate by Pclass + Sex heatmap
pivot = train.pivot_table(values='Survived', index='Sex', columns='Pclass', aggfunc='mean')
fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(pivot, annot=True, fmt='.1%', cmap='RdYlGn', vmin=0, vmax=1, ax=ax,
            linewidths=0.5)
ax.set_title('Survival Rate: Sex × Pclass')
plt.tight_layout()
plt.show()

## 5. Age Distribution by Survival

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE plot
for surv, label, color in [(0, 'Did Not Survive', '#e74c3c'), (1, 'Survived', '#2ecc71')]:
    subset = train[train['Survived'] == surv]['Age'].dropna()
    axes[0].hist(subset, bins=30, alpha=0.5, label=label, color=color, edgecolor='white')
axes[0].set_title('Age Distribution by Survival')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

# Violin plot
sns.violinplot(x='Survived', y='Age', data=train.dropna(subset=['Age']),
               palette=['#e74c3c', '#2ecc71'], inner='quartile', ax=axes[1])
axes[1].set_xticklabels(['Did Not Survive', 'Survived'])
axes[1].set_title('Age Distribution (Violin) by Survival')

plt.tight_layout()
plt.show()

print(f"Median age — Survived: {train[train['Survived']==1]['Age'].median():.1f}")
print(f"Median age — Not survived: {train[train['Survived']==0]['Age'].median():.1f}")

## 6. Fare Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall fare distribution (log scale)
axes[0].hist(train['Fare'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Fare Distribution')
axes[0].set_xlabel('Fare')
axes[0].set_ylabel('Count')

# Fare by Pclass
sns.boxplot(x='Pclass', y='Fare', data=train, palette='Blues', ax=axes[1])
axes[1].set_title('Fare by Passenger Class')
axes[1].set_xlabel('Passenger Class')
axes[1].set_ylabel('Fare')

plt.tight_layout()
plt.show()

print(train.groupby('Pclass')['Fare'].describe().round(2))

## 7. Correlation Heatmap

In [ ]:
# Encode categoricals for correlation
corr_df = train.copy()
corr_df['Sex_enc'] = (corr_df['Sex'] == 'female').astype(int)
corr_df['Embarked_enc'] = corr_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
num_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_enc', 'Embarked_enc']
corr_matrix = corr_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Family Size Analysis

In [ ]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Survival rate by family size
fs_surv = train.groupby('FamilySize')['Survived'].agg(['mean', 'count']).reset_index()
fs_surv.columns = ['FamilySize', 'SurvivalRate', 'Count']
bars = axes[0].bar(fs_surv['FamilySize'], fs_surv['SurvivalRate'],
                   color='steelblue', edgecolor='white')
axes[0].set_title('Survival Rate by Family Size')
axes[0].set_xlabel('Family Size (SibSp + Parch + 1)')
axes[0].set_ylabel('Survival Rate')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
# Annotate with count
for bar, (_, row) in zip(bars, fs_surv.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.01,
                 f'n={int(row["Count"])}', ha='center', fontsize=8)

# Alone vs. not alone
alone_surv = train.groupby('IsAlone')['Survived'].mean().reset_index()
sns.barplot(x='IsAlone', y='Survived', data=alone_surv,
            palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_xticklabels(['Travelling with Family', 'Travelling Alone'])
axes[1].set_title('Survival Rate: Alone vs. With Family')
axes[1].set_ylabel('Survival Rate')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()

## 9. Title Extraction Analysis

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from src.features import extract_title

train['Title'] = train['Name'].apply(extract_title)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Title counts
title_counts = train['Title'].value_counts()
sns.barplot(x=title_counts.index, y=title_counts.values, palette='viridis', ax=axes[0])
axes[0].set_title('Passenger Counts by Title')
axes[0].set_xlabel('Title')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Survival rate by title
title_surv = train.groupby('Title')['Survived'].mean().sort_values(ascending=False).reset_index()
sns.barplot(x='Title', y='Survived', data=title_surv, palette='coolwarm', ax=axes[1])
axes[1].set_title('Survival Rate by Title')
axes[1].set_xlabel('Title')
axes[1].set_ylabel('Survival Rate')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print('\nSurvival rate by title:')
print(train.groupby('Title')['Survived'].agg(['mean', 'count']).rename(
    columns={'mean': 'SurvivalRate', 'count': 'Count'}).round(3))